## Connect to Wharton Research Data Services (WRDS)

In [1]:
import wrds
import os
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
db = wrds.Connection(wrds_username=os.getenv("WRDS_USERNAME"))

Loading library list...
Done


In [16]:
crsp = db.list_tables(library='crsp')

In [6]:
dsfv2_info = db.describe_table(library="crsp", table="dsf_v2")
dsfv2_info

Approximately 110257376 rows in crsp.dsf_v2.


,name,nullable,type,comment
0,permno,True,INTEGER,PERMNO
1,hdrcusip,True,VARCHAR(8),Header CUSIP -8 Characters
2,permco,True,INTEGER,PERMCO
3,siccd,True,INTEGER,Sic Code
4,nasdissuno,True,INTEGER,Nasdaq Issue Number
5,yyyymmdd,True,INTEGER,YYYYMMDD - Daily Calendar Period Key
6,sharetype,True,VARCHAR(3),Share Type
7,securitytype,True,VARCHAR(4),Security Type
8,securitysubtype,True,VARCHAR(3),Security Sub-Type
9,usincflg,True,VARCHAR(1),US Incorporation Flag


### Get S&P500 Constituents

* Get all S&P constituents since 1957-03-04, when S&P 500 is officially created.
* Get their daily price during the time being.

(Note on SQL: inner join daily price of each stock in the given S&P500 period, left join their common name / ticker)

In [7]:
sp500 = db.raw_sql("""
    SELECT
        a.*,
        b.comnam,
        b.namedt,
        b.nameendt,
        b.ticker,
        c.dlycaldt AS date,
        c.dlyprc

    FROM crsp.msp500list AS a

    INNER JOIN crsp.dsf_v2 AS c
        ON a.permno = c.permno
        AND c.dlycaldt BETWEEN a.start AND a.ending
       

    LEFT JOIN crsp.msenames AS b
        ON a.permno = b.permno
        AND c.dlycaldt BETWEEN b.namedt AND b.nameendt

    WHERE c.dlycaldt >= DATE '1957-03-04'

""",
date_cols=[
    "start",
    "ending",
    "namedt",
    "nameendt",
    "date"
])

sp500

,permno,start,ending,comnam,namedt,nameendt,ticker,date,dlyprc
0,10006,1957-03-01,1984-07-18,A C F INDUSTRIES INC,1954-06-01,1962-07-01,<NA>,1957-03-04,60.75
1,10006,1957-03-01,1984-07-18,A C F INDUSTRIES INC,1954-06-01,1962-07-01,<NA>,1957-03-05,61.0
2,10006,1957-03-01,1984-07-18,A C F INDUSTRIES INC,1954-06-01,1962-07-01,<NA>,1957-03-06,60.25
3,10006,1957-03-01,1984-07-18,A C F INDUSTRIES INC,1954-06-01,1962-07-01,<NA>,1957-03-07,59.75
4,10006,1957-03-01,1984-07-18,A C F INDUSTRIES INC,1954-06-01,1962-07-01,<NA>,1957-03-08,59.375
...,...,...,...,...,...,...,...,...,...
44786,93436,2020-12-21,2024-12-31,TESLA INC,2024-06-20,2024-12-31,TSLA,2024-12-24,462.28
44787,93436,2020-12-21,2024-12-31,TESLA INC,2024-06-20,2024-12-31,TSLA,2024-12-26,454.13
44788,93436,2020-12-21,2024-12-31,TESLA INC,2024-06-20,2024-12-31,TSLA,2024-12-27,431.66
44789,93436,2020-12-21,2024-12-31,TESLA INC,2024-06-20,2024-12-31,TSLA,2024-12-30,417.41


In [9]:
sp500 = sp500.rename(columns={
    "dlyprc": "price",
    "comnam": "company_name",
    "namedt": "name_date",
    "nameendt": "end_name_date"
})
sp500 = sp500[
    [
        "permno",
        "date",
        "ticker",
        "price",
        "start",
        "ending",
        "company_name",
        "name_date",
        "end_name_date"
    ]
]
sp500

,permno,date,ticker,price,start,ending,company_name,name_date,end_name_date
0,10006,1957-03-04,<NA>,60.75,1957-03-01,1984-07-18,A C F INDUSTRIES INC,1954-06-01,1962-07-01
1,10006,1957-03-05,<NA>,61.0,1957-03-01,1984-07-18,A C F INDUSTRIES INC,1954-06-01,1962-07-01
2,10006,1957-03-06,<NA>,60.25,1957-03-01,1984-07-18,A C F INDUSTRIES INC,1954-06-01,1962-07-01
3,10006,1957-03-07,<NA>,59.75,1957-03-01,1984-07-18,A C F INDUSTRIES INC,1954-06-01,1962-07-01
4,10006,1957-03-08,<NA>,59.375,1957-03-01,1984-07-18,A C F INDUSTRIES INC,1954-06-01,1962-07-01
...,...,...,...,...,...,...,...,...,...
44786,93436,2024-12-24,TSLA,462.28,2020-12-21,2024-12-31,TESLA INC,2024-06-20,2024-12-31
44787,93436,2024-12-26,TSLA,454.13,2020-12-21,2024-12-31,TESLA INC,2024-06-20,2024-12-31
44788,93436,2024-12-27,TSLA,431.66,2020-12-21,2024-12-31,TESLA INC,2024-06-20,2024-12-31
44789,93436,2024-12-30,TSLA,417.41,2020-12-21,2024-12-31,TESLA INC,2024-06-20,2024-12-31


In [10]:
sp500.to_parquet(
    "../data/raw/sp500.parquet",
    index=False
)

In [14]:
sp500_membership = db.raw_sql("""
    SELECT
        a.*

    FROM crsp.msp500list AS a
""",
date_cols=[
    "start",
    "ending"
])

sp500_membership.to_parquet(
    "../data/raw/sp500_membership.parquet",
    index=False
)
sp500_membership

,permno,start,ending
0,10006,1957-03-01,1984-07-18
1,10030,1957-03-01,1969-01-08
2,10049,1925-12-31,1932-10-01
3,10057,1957-03-01,1992-07-02
4,10078,1992-08-20,2010-01-28
...,...,...,...
2059,93159,2012-07-31,2016-03-29
2060,93246,2021-03-22,2024-12-31
2061,93422,2010-07-01,2015-06-30
2062,93429,2017-03-01,2024-12-31
